# 12-03 RAG 深度问答

**高频考点**: Chunking 策略、Embedding 选择、检索优化、Reranking、评估指标

---

In [ ]:
# Q1: RAG 全流程
print("""
Q: 描述一个完整的 RAG Pipeline？

A: 离线索引 + 在线查询 两个阶段

  ===== 离线索引阶段 =====
  
  文档 → 解析 → 分块(Chunking) → Embedding → 向量数据库
  
  1. 文档解析: PDF/Word/HTML → 纯文本 (Unstructured/PyPDF2)
  2. 分块策略: RecursiveCharacterTextSplitter (chunk_size=512, overlap=64)
  3. Embedding: 文本 → 向量 (bge-large-zh / text-embedding-3-small)
  4. 存储: FAISS / Chroma / Milvus (含元数据: 来源、页码、时间)
  
  ===== 在线查询阶段 =====
  
  用户Query → 查询改写 → Embedding → 向量检索 → Reranking → LLM生成
  
  1. 查询改写: 口语→精确 query (LLM 改写 / HyDE)
  2. 检索: Dense(向量) + Sparse(BM25) 混合检索
  3. Reranking: 交叉编码器精排 top-20 → top-5
  4. Prompt组装: System + Context(检索结果) + Query
  5. LLM生成: 带引用来源的回答

  关键指标:
  - Retrieval: Recall@K, MRR, NDCG
  - Generation: Faithfulness, Relevance, Harmfulness
""")

In [ ]:
# Q2: Chunking 策略对比
print("""
Q: 常见的文档分块策略有哪些？如何选择？

A:
  ┌────────────────┬──────────────────┬──────────────┬──────────────┐
  │ 策略            │ 原理              │ 优点          │ 缺点          │
  ├────────────────┼──────────────────┼──────────────┼──────────────┤
  │ 固定大小        │ 按字符数切分       │ 简单、快速     │ 可能截断语义   │
  │ 递归分块        │ 按段落→句子→字符   │ 尽量保留语义   │ 需调参         │
  │                │ 逐级递归切分       │ (推荐默认方案) │               │
  │ 语义分块        │ 按 Embedding 相似  │ 语义完整性好   │ 计算开销大     │
  │                │ 度变化点切分       │               │               │
  │ 文档结构分块    │ 按标题/章节切分    │ 保留文档结构   │ 依赖格式规范   │
  │ 表格/代码分块   │ 特殊格式单独处理   │ 保留表格完整性 │ 需要特殊解析   │
  └────────────────┴──────────────────┴──────────────┴──────────────┘

  chunk_size 选择:
  - 太小 (128): 信息不完整，检索碎片化
  - 太大 (2048): 噪声多，Embedding 被稀释
  - 推荐: 512 tokens (兼顾完整性和精确性)

  overlap 选择:
  - 推荐 chunk_size 的 10-15% (e.g., 512→64 overlap)
  - 防止关键信息在边界被截断

  B站文档场景:
  - 帮助文档: 按标题分块 (文档结构分块)
  - FAQ: 每个 Q&A 作为一个 chunk
  - API 文档: 按 endpoint 分块
""")

# 代码演示: 不同 chunking 效果对比
sample_text = """
B站广告投放指南

一、开户流程
广告主需要先完成企业认证，提交营业执照、法人身份证等材料。审核通过后，
即可开通广告账户。开户费用为 5000 元起，其中包含首次充值金额。

二、广告形式
B站支持多种广告形式：
1. 开屏广告：APP启动时全屏展示，曝光量大
2. 信息流广告：融入推荐内容流，原生体验好
3. UP主商单：与UP主合作植入，用户接受度高

三、出价策略
支持 CPC（按点击付费）、CPM（按千次展示付费）、OCPC（智能出价）三种模式。
建议新广告主从 CPC 模式开始，积累数据后再切换到 OCPC。
"""

# 固定大小分块
def fixed_size_chunk(text, size=100, overlap=20):
    chunks = []
    for i in range(0, len(text), size - overlap):
        chunks.append(text[i:i+size].strip())
    return [c for c in chunks if c]

# 按段落分块 (简化版递归)
def paragraph_chunk(text):
    paragraphs = text.strip().split('\n\n')
    return [p.strip() for p in paragraphs if p.strip()]

print("=== 固定大小分块 (size=100) ===")
for i, chunk in enumerate(fixed_size_chunk(sample_text)):
    print(f"  Chunk {i}: {chunk[:60]}...")

print(f"\n=== 段落分块 ===")
for i, chunk in enumerate(paragraph_chunk(sample_text)):
    print(f"  Chunk {i}: {chunk[:60]}...")

In [ ]:
# Q3: Embedding 模型选择 & 向量检索 vs 关键词检索
print("""
Q: 如何选择 Embedding 模型？

A:
  中文场景:
  - bge-large-zh (BAAI): 中文 MTEB 排名靠前，768 维
  - m3e-base (Moka): 轻量级中文模型
  - text2vec-large-chinese: 中文语义相似度

  英文/多语言:
  - text-embedding-3-small (OpenAI): 1536维，效果好但需 API
  - all-MiniLM-L6-v2 (SentenceTransformers): 384维，轻量快速
  - bge-m3 (BAAI): 多语言支持，兼顾 dense/sparse/colbert

  选型原则:
  1. 语言匹配: 中文场景优先中文模型
  2. 维度 vs 性能: 维度越高越准但存储+检索越慢
  3. 成本: 本地部署 vs API 调用
  4. MTEB 排行榜作为参考

Q: 向量检索 vs 关键词检索 (BM25) 的区别？

A:
  向量检索 (Dense Retrieval):
  - 原理: 文本→向量，余弦相似度/内积
  - 擅长: 语义匹配 ("如何投放广告" ≈ "广告投放指南")
  - 弱点: 精确关键词匹配差 ("CPC出价" 可能检索不到 "CPC")
  
  关键词检索 (Sparse / BM25):
  - 原理: TF-IDF 变种，词频+逆文档频率
  - 擅长: 精确匹配、专有名词、代码
  - 弱点: 无法理解同义词、语义

  混合检索 (Hybrid, 推荐):
  - Dense + Sparse 结果融合
  - 融合方法: RRF (Reciprocal Rank Fusion)
    score = Σ 1/(k + rank_i)  (k=60)
  - 通常比单一检索提升 10-20% Recall
""")

# 代码演示: 简单向量检索 vs 关键词检索
import numpy as np

# 模拟 Embedding (简化为随机向量)
np.random.seed(42)
docs = [
    "B站广告投放支持CPC和CPM两种计费模式",
    "UP主可以通过花火平台接商业合作",
    "信息流广告融入推荐内容获得更好的用户体验",
    "开屏广告在APP启动时全屏展示品牌信息",
    "OCPC智能出价会自动优化转化成本"
]

# 简单关键词匹配
def keyword_search(query, docs):
    scores = []
    query_words = set(query)
    for doc in docs:
        # 简单字符匹配
        score = sum(1 for w in query_words if w in doc)
        scores.append(score)
    ranked = sorted(enumerate(scores), key=lambda x: -x[1])
    return ranked

query = "CPC出价模式"
print(f"Query: {query}")
print("\n关键词检索结果:")
for idx, score in keyword_search(query, docs)[:3]:
    print(f"  [{score}分] {docs[idx]}")

print("\n→ 混合检索 = 向量(语义) + 关键词(精确) 互补，效果最佳")

In [ ]:
# Q4: Reranking & 高级 RAG 技术
print("""
Q: 什么是 Reranking？为什么需要？

A:
  检索分两阶段:
  1. 召回 (Retrieval): 从百万文档中快速召回 top-100 (向量近似搜索, ~10ms)
  2. 精排 (Reranking): 对 top-100 用更精确的模型重排序 → top-5

  Reranker 模型:
  - 交叉编码器 (Cross-Encoder): 同时编码 query+doc，精度高但慢
  - bge-reranker-large: 开源中文 reranker
  - Cohere Rerank: API 方式，效果好
  
  为什么需要: 向量检索是"双塔模型" (query和doc分别编码)，信息交互少;
  Reranker 是"单塔模型" (query+doc一起编码)，交互充分，精度更高。
  典型提升: Recall@5 提升 10-15%

Q: 高级 RAG 技术有哪些？

A:
  1. HyDE (Hypothetical Document Embedding):
     - 原理: 先让 LLM 生成假设性答案，用答案的 embedding 去检索
     - 效果: query 和 doc 在语义空间更接近
     - 适合: 用户 query 较短/模糊时
  
  2. Self-RAG:
     - 原理: LLM 自己判断 "是否需要检索" → "检索结果是否相关" → "回答是否忠实"
     - 效果: 减少不必要检索，提高生成忠实度
     - 实现: 在生成中插入 special tokens 做自我反思
  
  3. Graph-RAG (微软):
     - 原理: 文档→知识图谱→社区检测→社区摘要→多层次检索
     - 效果: 回答需要全局视角的问题 ("B站所有广告形式的优缺点")
     - 适合: 需要跨文档推理、汇总性问题
  
  4. Corrective RAG (CRAG):
     - 原理: 检索后评估文档相关性，不相关则触发 web 搜索
     - 效果: 动态补充知识源

  5. Agentic RAG:
     - 原理: 用 Agent 编排 RAG 流程，动态决定检索策略
     - 效果: 根据 query 复杂度自适应选择 RAG 方案
""")

# 代码演示: HyDE 原理 (模拟)
print("=" * 50)
print("HyDE 示例 (模拟)")
print("=" * 50)

query = "B站广告怎么投？"
hypothetical_answer = """
B站广告投放流程：首先需要在B站广告平台注册账号并完成企业认证，
提交营业执照等资质材料。审核通过后充值开户，然后选择广告形式
（信息流、开屏、搜索等），设置定向人群、出价策略和预算，
上传素材后提交审核，审核通过即可开始投放。
"""

print(f"原始 Query: {query}")
print(f"HyDE 生成的假设答案: {hypothetical_answer.strip()[:80]}...")
print("→ 用假设答案的 Embedding 去检索，比短 query 的 Embedding 更准确")

In [ ]:
# Q5: RAG 评估指标
print("""
Q: 如何评估 RAG 系统的效果？

A: 三个层面的评估

  ===== 1. 检索评估 (Retrieval) =====
  
  Recall@K: 前 K 个结果中包含正确文档的比例
  - Recall@5 = 正确文档出现在 top-5 中的概率
  
  MRR (Mean Reciprocal Rank): 第一个正确结果的排名倒数的平均
  - 正确结果排第1 → 1/1 = 1.0
  - 正确结果排第3 → 1/3 = 0.33
  
  NDCG: 考虑相关性等级的排名质量

  ===== 2. 生成评估 (Generation) =====
  
  Faithfulness (忠实度): 回答是否基于检索到的文档
  - 评估: LLM-as-Judge 检查回答中每个声明是否有文档支撑
  
  Relevance (相关性): 回答是否切题
  - 评估: LLM-as-Judge 打分 1-5
  
  Harmfulness (安全性): 回答是否包含有害内容

  ===== 3. 端到端评估 =====
  
  Golden Dataset: 人工标注的 QA pair
  - 100-200 条高质量标注
  - 定期回归测试，防止退化
  
  A/B 测试: 上线后对比新旧系统的用户满意度

  ===== 评估工具 =====
  - RAGAS: 开源 RAG 评估框架 (Faithfulness, Relevance, Recall)
  - LangSmith: 在线评估 + 链路追踪
  - Phoenix (Arize): 可视化评估
""")

# 代码演示: 简单 Recall@K 计算
def recall_at_k(retrieved_ids, relevant_ids, k):
    """计算 Recall@K"""
    top_k = set(retrieved_ids[:k])
    relevant = set(relevant_ids)
    hits = top_k & relevant
    return len(hits) / len(relevant) if relevant else 0.0

def mrr(retrieved_ids, relevant_ids):
    """计算 MRR"""
    relevant = set(relevant_ids)
    for rank, doc_id in enumerate(retrieved_ids, 1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

# 模拟评估
print("=" * 50)
print("RAG 检索评估示例")
print("=" * 50)

# 假设检索返回的文档 ID 和真正相关的文档 ID
retrieved = [3, 7, 1, 5, 2, 8, 4, 6]  # 检索结果排序
relevant = [1, 5]  # 真正相关的文档

for k in [1, 3, 5]:
    r = recall_at_k(retrieved, relevant, k)
    print(f"  Recall@{k} = {r:.2f}")

m = mrr(retrieved, relevant)
print(f"  MRR = {m:.2f} (第一个相关文档排在第 {retrieved.index(relevant[0])+1} 位)")

## 面试速查卡片

| 题目 | 一句话回答 |
|------|-----------|
| RAG 全流程 | 文档→分块→Embedding→向量库 (离线); Query→改写→检索→Rerank→生成 (在线) |
| Chunk size 怎么选 | 512 tokens 为默认起点，overlap=10-15%，实验对比决定 |
| Embedding 选型 | 中文: bge-large-zh; 英文: text-embedding-3-small; 参考 MTEB 排行榜 |
| 向量 vs 关键词检索 | 向量擅长语义，BM25 擅长精确匹配，混合检索 (RRF融合) 效果最佳 |
| 为什么要 Reranking | 召回用"双塔"快但粗，精排用"单塔"慢但准，Recall 提升 10-15% |
| HyDE 原理 | LLM 先生成假设答案，用答案的 embedding 检索，弥补短 query 信息不足 |
| Graph-RAG 适合什么 | 需要跨文档推理、全局汇总的问题（如对比分析、趋势总结） |
| RAG 评估三层 | Retrieval(Recall/MRR) + Generation(Faithfulness/Relevance) + E2E(Golden Dataset) |